# MyDigitalTwin — Analyse Fréquentielle des Centres d'Intérêt

**Objectif** : Identifier les concepts les plus fréquents dans mes données réelles (Spotify, YouTube, Google, Netflix, Chrome) pour calibrer les macro-catégories de la home page.

**Approche** : Analyse de fréquence de mots par source → identification des gaps dans `CATEGORY_KEYWORDS` → mise à jour du dictionnaire.

> **Pourquoi pas K-Means ?**  
> Une première tentative de clustering TF-IDF + K-Means a produit un cluster *catch-all* dominant (~73% des données, Silhouette ≈ 0.19). Le problème est structurel : les textes courts multi-sources (artistes Spotify, titres YouTube, requêtes Google) ont des espaces sémantiques trop hétérogènes pour être clusterisés conjointement.  
> L'analyse fréquentielle est plus directe et interprétable pour des catégories prédéfinies.

In [1]:
# ── 0. SETUP ──────────────────────────────────────────────────────────────────
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("MyDigitalTwin-FrequencyAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# ── Config centrale ────────────────────────────────────────────────────────────
import sys as _sys, os as _os
_sys.path.insert(0, _os.path.abspath(_os.path.join(_os.path.dirname('__file__'), '../../..')))
from config import WAREHOUSE

print(f"Warehouse: {WAREHOUSE}")
assert os.path.exists(WAREHOUSE), f"Warehouse introuvable: {WAREHOUSE}"

def read_table(table_name):
    return spark.read.parquet(os.path.join(WAREHOUSE, table_name))

Warehouse: /opt/spark/warehouse


26/04/05 12:26:24 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


---
## Étape 1 — Chargement des sources texte

On extrait la colonne texte pertinente de chaque source Delta, en gardant la provenance (`source`) pour analyser chaque canal séparément.

In [2]:
# ── 1. CHARGEMENT ─────────────────────────────────────────────────────────────
sources = {
    "Google Searches": read_table("google_searches").select(F.col("query").alias("text")),
    "YouTube":         read_table("youtube_watch").select(F.col("title").alias("text")),
    "Chrome":          read_table("google_chrome").select(F.col("title").alias("text")),
    "Spotify":         read_table("spotify_streams").select(F.col("artistName").alias("text")).dropDuplicates(["text"]),
    "Netflix":         read_table("netflix_views").select(F.col("show_title").alias("text")),
}

for name, df in sources.items():
    print(f"{name:20s}: {df.count():>6,} lignes")

Google Searches     : 55,854 lignes
YouTube             : 13,821 lignes
Chrome              :    338 lignes


Spotify             :  2,924 lignes
Netflix             :  4,288 lignes


In [3]:
# ── 2. ANALYSE FRÉQUENTIELLE PAR SOURCE ───────────────────────────────────────
from pyspark.ml.feature import Tokenizer, StopWordsRemover

# Stopwords : anglais + français + bruit technique
STOPWORDS_EXTRA = [
    # Français
    "les", "des", "une", "sur", "avec", "dans", "qui", "que", "par", "plus",
    "tout", "bien", "comme", "mais", "mon", "ton", "son", "nos", "mes",
    "faire", "comment", "plus", "aussi", "encore", "très",
    # Anglais générique
    "the", "and", "for", "with", "you", "your", "this", "that", "from",
    "are", "was", "not", "its", "but", "all", "new", "best", "how",
    # Bruit technique (URLs, tracking, ads)
    "https", "http", "www", "com", "org", "net", "html", "php", "utm",
    "amp", "utm_source", "befr", "dgoogle", "watch", "video", "clip",
    "official", "officiel", "youtube", "shorts",
]

STOP_ALL = StopWordsRemover.loadDefaultStopWords("english") \
         + StopWordsRemover.loadDefaultStopWords("french") \
         + STOPWORDS_EXTRA

results = {}

for source_name, df in sources.items():
    clean = df.filter(
        F.col("text").isNotNull() &
        (F.length(F.col("text")) > 2) &
        (~F.col("text").rlike(r'^https?://'))   # exclure les URLs brutes
    )

    tokenized = Tokenizer(inputCol="text", outputCol="words").transform(clean)
    filtered  = StopWordsRemover(
        inputCol="words", outputCol="tokens", stopWords=STOP_ALL
    ).transform(tokenized)

    freq = (
        filtered
        .select(F.explode("tokens").alias("word"))
        .filter(F.length("word") > 2)
        .groupBy("word")
        .count()
        .orderBy(F.desc("count"))
    )

    results[source_name] = freq
    print(f"✓ {source_name}")

print("\nAnalyse terminée.")

✓ Google Searches
✓ YouTube
✓ Chrome
✓ Spotify
✓ Netflix

Analyse terminée.


In [4]:
# ── 3. TOP 25 MOTS PAR SOURCE ─────────────────────────────────────────────────
TOP_N = 25

for source_name, freq_df in results.items():
    print(f"\n{'='*50}")
    print(f"  {source_name}")
    print(f"{'='*50}")
    freq_df.show(TOP_N, truncate=False)


  Google Searches


+----------+-----+
|word      |count|
+----------+-----+
|belgique  |312  |
|fifa      |266  |
|one       |261  |
|google    |242  |
|streaming |227  |
|piece     |203  |
|minecraft |203  |
|%c3%a0    |189  |
|prix      |177  |
|hannut    |161  |
|fortnite  |161  |
|synonyme  |155  |
|musique   |149  |
|mac       |146  |
|discord   |144  |
|pdf       |140  |
|download  |136  |
|film      |136  |
|traduction|135  |
|pro       |135  |
|carte     |135  |
|font      |133  |
|mp3       |132  |
|france    |129  |
|club      |127  |
+----------+-----+
only showing top 25 rows


  YouTube
+----------+-----+
|word      |count|
+----------+-----+
|16x9      |571  |
|inh       |508  |
|officiel) |310  |
|(clip     |309  |
|mix       |256  |
|(official |241  |
|video)    |234  |
|vid       |215  |
|music     |196  |
|live      |194  |
|2024      |180  |
|1920x1080 |178  |
|2025      |170  |
|15s       |169  |
|house     |164  |
|ft.       |160  |
|(ft       |153  |
|squeezie  |153  |
|one       |1

In [5]:
# ── 4. BIGRAMMES — Termes composés importants ─────────────────────────────────
# Les bigrammes capturent des concepts que les mots seuls manquent :
# "travis scott", "league of legends", "formula 1", "deep learning", etc.
from pyspark.ml.feature import NGram

bigram_results = {}

for source_name, df in sources.items():
    clean = df.filter(
        F.col("text").isNotNull() &
        (F.length(F.col("text")) > 2) &
        (~F.col("text").rlike(r'^https?://'))
    )

    tokenized = Tokenizer(inputCol="text", outputCol="words").transform(clean)
    filtered  = StopWordsRemover(
        inputCol="words", outputCol="tokens", stopWords=STOP_ALL
    ).transform(tokenized)

    bigrams = NGram(n=2, inputCol="tokens", outputCol="ngrams").transform(filtered)

    freq = (
        bigrams
        .select(F.explode("ngrams").alias("bigram"))
        .filter(F.length("bigram") > 5)
        .groupBy("bigram")
        .count()
        .orderBy(F.desc("count"))
    )

    bigram_results[source_name] = freq

print("Top bigrammes par source :\n")
for source_name, freq_df in bigram_results.items():
    print(f"── {source_name}")
    freq_df.show(15, truncate=False)
    print()

Top bigrammes par source :

── Google Searches


+---------------+-----+
|bigram         |count|
+---------------+-----+
|one piece      |189  |
|fifa 21        |152  |
|fifa 23        |59   |
|ralph lauren   |46   |
|streaming vf   |42   |
|travis scott   |42   |
|epic games     |39   |
|airpods pro    |39   |
|google flight  |36   |
|virtual dj     |36   |
|freeze corleone|36   |
|rocket league  |35   |
|polo ralph     |33   |
|stg gege       |33   |
|star wars      |32   |
+---------------+-----+
only showing top 15 rows


── YouTube
+--------------------+-----+
|bigram              |count|
+--------------------+-----+
|(clip officiel)     |304  |
|16x9 6s             |234  |
|inh inh             |196  |
|vid 16x9            |196  |
|choisissez chrome   |114  |
|- rediffusion       |106  |
|rediffusion squeezie|106  |
|travis scott        |103  |
|inh cards           |102  |
|music video)        |100  |
|(official video)    |99   |
|inazuma eleven      |98   |
|(official music     |93   |
|eleven -            |85   |
|- saison    

In [6]:
# ── 5. GAP ANALYSIS — Termes fréquents non couverts par CATEGORY_KEYWORDS ─────
# On identifie les mots qui apparaissent souvent mais ne sont dans aucune catégorie.

CATEGORY_KEYWORDS = {
    "Sport":          ["football","soccer","nba","match","goal","arsenal","fifa","ligue","rugby",
                       "tennis","basketball","sport","ucl","premier league","ufc","mma","boxing",
                       "gym","fitness","workout","calisthenics","training","nfl","olympics",
                       "swimming","cycling","padel","volleyball","champions league","ligue 1"],
    "Auto/Moto":      ["car","auto","voiture","porsche","ferrari","lamborghini","bmw","mercedes",
                       "audi","tesla","f1","formula 1","supercar","hypercar","drift","tuning",
                       "engine","motorsport","motorcycle","moto","yamaha","kawasaki","mclaren",
                       "bugatti","ducati","harley","grand prix","jdm","supra","amg"],
    "Musique":        ["music","song","artist","rap","album","track","beat","drill","trap",
                       "afrobeat","afropop","rnb","r&b","hip","hop","spotify","playlist",
                       "concert","festival","lyrics","producer","techno","house","electro",
                       "lo-fi","jazz","dj","remix","soundcloud","damso","tiakola","ninho",
                       "zamdane","gazo","niska","bezbar","travis scott","freeze corleone"],
    "Tech":           ["python","code","data","dev","javascript","api","ai","software","tech",
                       "developer","engineering","technology","artificial intelligence",
                       "machine learning","deep learning","nlp","llm","pytorch","tensorflow",
                       "backend","frontend","react","github","docker","kubernetes","cloud",
                       "aws","cybersecurity","linux","startup","chatgpt","openai","gpu","nvidia"],
    "Cinema/Series":  ["netflix","film","série","movie","episode","cinema","trailer","season",
                       "streaming","anime","manga","hbo","marvel","star wars","oscars",
                       "naruto","shippuden","fairy tail","jojo","baki","boruto","fullmetal",
                       "hunter","ghibli","rick morty","one piece"],
    "Gaming":         ["game","gaming","xbox","ps5","steam","minecraft","fortnite","esport",
                       "gamer","nintendo","switch","twitch","discord","multiplayer","rpg",
                       "fps","roblox","league of legends","valorant","warzone","gta",
                       "elden ring","zelda","playstation"],
    "Actu/Societe":   ["news","actu","monde","france","afrique","africa","belgique","politique",
                       "environment","ecology","climate","space","nasa","spacex","economy",
                       "finance","crypto","bitcoin","blockchain","stock market","history",
                       "philosophy"],
    "Shopping":       ["amazon","shop","brand","adidas","nike","fashion","streetwear","sneakers",
                       "yeezy","jordan","clothes","outfit","ecommerce","unboxing","skincare",
                       "watches","apple","iphone","samsung","gadget","dior","louis vuitton"],
    "Photo/Crea":     ["photo","photography","design","creative","art","visual","camera",
                       "graphic","illustration","photoshop","lightroom","editing",
                       "content creation","tiktok","reels","architecture","digital art",
                       "ui/ux","3d modeling","blender","canva"],
}

all_kw = {kw for kws in CATEGORY_KEYWORDS.values() for kw in kws}

# Union de toutes les sources pour la vue globale
from functools import reduce
all_sources = reduce(lambda a, b: a.union(b), sources.values())
clean_all = all_sources.filter(
    F.col("text").isNotNull() &
    (F.length(F.col("text")) > 2) &
    (~F.col("text").rlike(r'^https?://'))
)
tokenized_all = Tokenizer(inputCol="text", outputCol="words").transform(clean_all)
filtered_all  = StopWordsRemover(inputCol="words", outputCol="tokens", stopWords=STOP_ALL).transform(tokenized_all)

global_freq = (
    filtered_all
    .select(F.explode("tokens").alias("word"))
    .filter(F.length("word") > 2)
    .groupBy("word").count()
    .orderBy(F.desc("count"))
)

top_words = [row["word"] for row in global_freq.limit(200).collect()]
uncovered = [w for w in top_words if w not in all_kw]

print("Top 40 mots fréquents NON couverts par CATEGORY_KEYWORDS :")
print("(candidats à ajouter dans une catégorie)\n")
for w in uncovered[:40]:
    print(f"  {w}")

Top 40 mots fréquents NON couverts par CATEGORY_KEYWORDS :
(candidats à ajouter dans une catégorie)

  16x9
  inh
  one
  piece
  google
  officiel)
  (clip
  mix
  pro
  (official
  video)
  live
  2024
  vid
  club
  prix
  saison
  2025
  squeezie
  mac
  %c3%a0
  musique
  1920x1080
  15s
  download
  hannut
  ft.
  travis
  jbl
  black
  synonyme
  show
  (ft
  scott
  fairy
  tail
  jeu
  (2011)
  chrome
  pdf


In [7]:
spark.stop()
print("Spark session fermée.")

Spark session fermée.
